In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
try:
    df = pd.read_csv('voice.csv')

    # Check if 'label' is the target column
    if 'label' in df.columns:
        # Encode label
        le = LabelEncoder()
        df['label_encoded'] = le.fit_transform(df['label'])

        X = df.drop(['label', 'label_encoded'], axis=1)
        y = df['label_encoded']

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Scale features (KNN is sensitive to feature scale)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Find optimal K
        error_rate = []
        for i in range(1, 21):
            knn = KNeighborsClassifier(n_neighbors=i)
            knn.fit(X_train_scaled, y_train)
            pred_i = knn.predict(X_test_scaled)
            error_rate.append(np.mean(pred_i != y_test))

        best_k = error_rate.index(min(error_rate)) + 1

        # Train final model
        knn_final = KNeighborsClassifier(n_neighbors=best_k)
        knn_final.fit(X_train_scaled, y_train)
        y_pred = knn_final.predict(X_test_scaled)

        # Evaluate
        acc = accuracy_score(y_test, y_pred)
        report = classification_report(y_test, y_pred, target_names=le.classes_)

        # Confusion matrix plot
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title(f'Confusion Matrix (k={best_k})')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        plt.savefig('cm_knn.png')
        plt.close()

        print(f"Dataset shape: {df.shape}")
        print(f"Optimal k value: {best_k}")
        print(f"Accuracy: {acc:.4f}")
        print("\nClassification Report:")
        print(report)
    else:
        print("Target column 'label' not found. Columns are:", df.columns.tolist())
except Exception as e:
    print(f"Error: {e}")

Dataset shape: (3168, 22)
Optimal k value: 7
Accuracy: 0.9826

Classification Report:
              precision    recall  f1-score   support

      female       0.99      0.98      0.98       297
        male       0.98      0.99      0.98       337

    accuracy                           0.98       634
   macro avg       0.98      0.98      0.98       634
weighted avg       0.98      0.98      0.98       634



In [2]:
# Load data
df = pd.read_csv('voice.csv')

# Encode label
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculate error rates for different K values
error_rates = []
k_values = range(1, 31)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred_k = knn.predict(X_test_scaled)
    error_rates.append(np.mean(pred_k != y_test))

# Find the best K
best_k = k_values[error_rates.index(min(error_rates))]
min_error = min(error_rates)

# Plot Error Rate vs K
plt.figure(figsize=(10, 6))
plt.plot(k_values, error_rates, color='blue', linestyle='dashed', marker='o',
         markerfacecolor='red', markersize=8)
plt.title('Tingkat Error vs. Nilai K')
plt.xlabel('Nilai K')
plt.ylabel('Tingkat Error (Error Rate)')
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.7)

# Highlight best K
plt.axvline(x=best_k, color='green', linestyle=':', label=f'Best k={best_k}')
plt.legend()
plt.tight_layout()
plt.savefig('k_analysis.png')
plt.close()

print(f"Best K: {best_k}")
print(f"Minimum Error Rate: {min_error:.4f}")

Best K: 7
Minimum Error Rate: 0.0174
